# Technical Webpage Summarizer

Week 1 community contribution: a standalone website summarizer using **NVIDIA NIM** through the OpenAI-compatible API.

Give it a URL. It fetches the page, strips navigation and other noise, and returns a short, objective technical summary.


## 1. Imports

Load the libraries used in this notebook.


In [1]:
import os
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI


## 2. Environment & NVIDIA API Key

Load `NVIDIA_API_KEY` from a `.env` file. Do not paste the key into this notebook.


In [2]:
load_dotenv(override=True)

api_key = os.getenv("NVIDIA_API_KEY")

if not api_key:
    print("NVIDIA_API_KEY was not found. Add it to your .env file.")
elif api_key.strip() != api_key:
    print("NVIDIA_API_KEY has leading or trailing whitespace.")
else:
    print("NVIDIA_API_KEY loaded.")


NVIDIA_API_KEY loaded.


## 3. Connect to NVIDIA NIM

Use the OpenAI Python client with NVIDIA's OpenAI-compatible endpoint.


In [3]:
NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"
MODEL = "nvidia/nemotron-3.5-lightning-30b-a3b"

client = OpenAI(
    base_url=NVIDIA_BASE_URL,
    api_key=api_key,
)


## 4. Fetch Website Content

Download the page with `requests`, parse it with BeautifulSoup, and keep the main readable text.

Scripts, styles, navigation, headers, footers, ads, cookie banners, and similar noise are removed where possible.


In [4]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/117.0.0.0 Safari/537.36"
    )
}

IRRELEVANT_TAGS = [
    "script",
    "style",
    "noscript",
    "nav",
    "header",
    "footer",
    "aside",
    "form",
    "iframe",
    "svg",
    "img",
    "input",
    "button",
]

NOISE_KEYWORDS = (
    "nav",
    "menu",
    "header",
    "footer",
    "cookie",
    "consent",
    "banner",
    "advert",
    "ads",
    "promo",
    "sidebar",
    "social",
    "newsletter",
)


def _attr_text(element, key: str) -> str:
    if getattr(element, "attrs", None) is None:
        return ""
    value = element.get(key)
    if not value:
        return ""
    if isinstance(value, list):
        return " ".join(str(item) for item in value if item)
    return str(value)


def _looks_like_noise(element) -> bool:
    if getattr(element, "attrs", None) is None:
        return False
    attrs = " ".join(
        filter(
            None,
            [
                _attr_text(element, "id"),
                _attr_text(element, "class"),
                _attr_text(element, "role"),
                _attr_text(element, "aria-label"),
            ],
        )
    ).lower()
    return any(keyword in attrs for keyword in NOISE_KEYWORDS)


def fetch_website_content(url: str) -> str:
    """Return the page title plus cleaned body text."""
    response = requests.get(url, headers=HEADERS, timeout=20)
    response.raise_for_status()

    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.get_text(strip=True) if soup.title else "No title found"

    root = soup.body if soup.body else soup
    for tag in root.find_all(IRRELEVANT_TAGS):
        if getattr(tag, "attrs", None) is None:
            continue
        tag.decompose()

    for tag in root.find_all(True):
        if getattr(tag, "attrs", None) is None:
            continue
        if _looks_like_noise(tag):
            tag.decompose()

    text = root.get_text(separator="\n", strip=True)
    return f"{title}\n\n{text}"


## 5. Create Prompt

The system prompt asks for a short, clear, objective technical summary, plus a mechanism-design relevance section when the source supports it. The user prompt provides the cleaned page text.


In [5]:
system_prompt = """
You are a helpful technical research assistant.

Analyze the contents of a webpage and produce a short, clear, objective summary.

Focus on:
- What the page is about
- The main technical points
- Important specifications or features
- Who the information is useful for
- Any practical takeaways

After the general summary, include this section:

## Relevance to Mechanism Design
### Mekanizma Tasarımı Açısından Önemi

This section is for linkage and mechanism-design research, including six-bar and Watt-type mechanisms.

If the page is technical or academic, extract the following only when the source states them. If a field is absent, write "not specified" / "belirtilmemiş". Never invent values that are not in the source.

- Mechanism topology
- Degrees of freedom (DOF)
- Synthesis method
- Input and output motion
- Fingertip or end-effector trajectory
- Orientation tracking
- Link-length information
- Transmission or force-related evaluation
- Verification / test method
- Limitations
- Practical engineering implications

If the page is not about mechanisms, keep this section brief and mark mechanism-specific fields as "not specified" / "belirtilmemiş".

Ignore navigation menus, cookie banners, ads, login prompts, and unrelated page elements.

Keep the tone objective and technical. Respond in markdown. Do not wrap the markdown in a code block.
""".strip()


def user_prompt_for(website_content: str) -> str:
    return (
        "Here are the contents of a website.\n"
        "Provide a short, clear, objective technical summary.\n"
        "If it includes news or announcements, summarize those too.\n\n"
        f"{website_content}"
    )


## 6. Build Messages

NVIDIA NIM expects OpenAI-style chat messages: a system message plus a user message.


In [6]:
def messages_for(website_content: str) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website_content)},
    ]


## 7. Summarize Website

Fetch the page, build messages, and ask NVIDIA NIM for a summary.


In [7]:
def summarize(url: str) -> str:
    website_content = fetch_website_content(url)
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages_for(website_content),
    )
    return response.choices[0].message.content


## 8. Display Summary

Render the model output as markdown in the notebook.


In [8]:
def display_summary(url: str) -> None:
    summary = summarize(url)
    display(Markdown(summary))


## 9. Example

Summarize [https://www.nvidia.com](https://www.nvidia.com).

Some sites are JavaScript-heavy and return little HTML to `requests`. If the summary looks thin, try another URL.


In [9]:
display_summary(
    "https://scholar.dgist.ac.kr/handle/20.500.11750/5078"
)

**Technical Summary**

The page is a 2016 academic article (*Mechanism and Machine Theory*, v.104, pp.177–189) titled “Synthesis of a Watt II six-bar linkage in the design of a hand rehabilitation robot.” It proposes a kinematic synthesis framework for a six-bar linkage used in a hand exoskeleton robot that targets finger rehabilitation in patients with neurological disabilities (stroke, spinal cord injury). The design incorporates real grasping motion data from the human forefinger to drive the linkage synthesis. The synthesis employs a two-loop approach: geometrical synthesis of the first loop achieves continuous input rotation, while body-guidance synthesis of the second loop directs the end-effector to follow desired trajectories while maintaining target orientations. The work addresses the need for repeatable, adjustable therapeutic training in robotic rehabilitation, though detailed prototype validation or test methods are not described in the abstract. It contributes to mechanism design for assistive robotics by linking real human motion data to six-bar kinematics.

**Useful for:** Researchers and engineers in robotic rehabilitation, mechanism design, and kinematic synthesis of assistive exoskeletons.

**Practical takeaway:** A two-stage synthesis method (geometrical + body-guidance) can translate real grasping motions into six-bar linkage parameters, enabling continuous input rotation and oriented trajectory tracking for hand rehabilitation robots.

## Relevance to Mechanism Design
### Mekanizma Tasarımı Açısından Önemi

- **Mechanism topology:** Watt II six-bar mechanism (as stated in the source)
- **Degrees of freedom (DOF):** not specified / belirtilmemiş
- **Synthesis method:** Geometrical synthesis used for the first loop to attain continuous input rotation; body-guidance synthesis used for the second loop for end-effector trajectory and orientation tracking (as stated)
- **Input and output motion:** Input provides continuous rotation (first loop); output steers the end-effector to follow desired trajectories in desired orientations (second loop)
- **Fingertip or end-effector trajectory:** desired trajectory in desired orientations (as stated)
- **Orientation tracking:** desired orientations (as stated)
- **Link-length information:** not specified / belirtilmemiş
- **Transmission or force-related evaluation:** not specified / belirtilmemiş
- **Verification / test method:** not specified / belirtilmemiş
- **Limitations:** not specified / belirtilmemiş
- **Practical engineering implications:** Enables kinematic synthesis of six-bar linkages for hand rehabilitation robots using real human grasping motion data; supports repeatable, variable therapeutic training for neurological disability patients; demonstrates a two-loop synthesis approach linking geometrical and body-guidance methods for continuous input and oriented trajectory tracking.